# 00 - Colab Setup and Data Check

This notebook prepares the Colab runtime and verifies that the real labeled, real unlabeled, and synthetic datasets are available before running any training experiments.

It does not train models.

## 1. Mount Google Drive

Mount Drive so synthetic images and experiment outputs can be read/written from persistent storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull The Repository

If the repository already exists in the runtime, pull the latest changes. Otherwise, clone it.

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    print(f'Repository exists at {REPO_ROOT}. Pulling latest changes...')
    %cd {REPO_ROOT}
    !git pull
else:
    print(f'Cloning repository to {REPO_ROOT}...')
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('Current working directory:', Path.cwd())

## 3. Install Dependencies

Install from `requirements.txt` when it is available. This notebook only needs lightweight packages for verification, but installing the project dependencies early makes later notebooks more predictable.

In [ ]:
requirements_path = REPO_ROOT / 'requirements.txt'

if requirements_path.exists():
    print(f'Installing dependencies from {requirements_path}...')
    !pip install -q -r {requirements_path}
else:
    print('WARNING: requirements.txt not found. Skipping dependency installation.')

## 4. Editable Paths

Update `DCGAN_SYNTHETIC_DIR` and `ACGAN_SYNTHETIC_DIR` to point to your Google Drive folders containing generated images. The expected class folders are `COVID`, `Lung_Opacity`, `Viral_Pneumonia`, and `Normal`.

ACGAN is kept for Stage 1 synthetic quality comparison. DCGAN is the default synthetic source for the report's `COVID-QU-Syn` downstream contrastive pretraining and classification experiments.

The defaults follow `configs/experiments/common.yaml` and `configs/experiments/paths.template.yaml`.

In [ ]:
# Edit these paths for your Colab/Drive layout.
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
REAL_LABELED_DIR = REPO_ROOT / 'data/processed/labelled_4232'
REAL_UNLABELED_DIR = REPO_ROOT / 'data/processed/unlabelled_16934'

# Replace these placeholders with your Drive folders for synthetic images.
DCGAN_SYNTHETIC_DIR = Path('/content/drive/MyDrive/path/to/DCGAN_synthetic_images')
ACGAN_SYNTHETIC_DIR = Path('/content/drive/MyDrive/path/to/ACGAN_synthetic_images')

# Store notebook outputs and later experiment results on Drive.
OUTPUT_ROOT = Path('/content/drive/MyDrive/contrastive-synthesis-medcls_CVProject/results/experiments')

CLASS_NAMES = ['COVID', 'Lung_Opacity', 'Viral_Pneumonia', 'Normal']
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

print('REPO_ROOT:', REPO_ROOT)
print('REAL_LABELED_DIR:', REAL_LABELED_DIR)
print('REAL_UNLABELED_DIR:', REAL_UNLABELED_DIR)
print('DCGAN_SYNTHETIC_DIR:', DCGAN_SYNTHETIC_DIR)
print('ACGAN_SYNTHETIC_DIR:', ACGAN_SYNTHETIC_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 5. Helper Functions

These helpers count images, validate class folders, and display samples without assuming every optional path is present.

In [ ]:
from collections import OrderedDict
import random
from PIL import Image
import matplotlib.pyplot as plt

def is_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS

def find_images(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob('*') if is_image(p)])

def class_image_dir(dataset_root: Path, class_name: str) -> Path:
    class_root = dataset_root / class_name
    image_subdir = class_root / 'images'
    return image_subdir if image_subdir.exists() else class_root

def count_class_images(dataset_root: Path, class_names=CLASS_NAMES):
    counts = OrderedDict()
    for class_name in class_names:
        counts[class_name] = len(find_images(class_image_dir(dataset_root, class_name)))
    return counts

def validate_class_names(dataset_root: Path, dataset_name: str, required=CLASS_NAMES):
    if not dataset_root.exists():
        print(f'WARNING: {dataset_name} path does not exist: {dataset_root}')
        return False

    present = sorted([p.name for p in dataset_root.iterdir() if p.is_dir()])
    missing = [c for c in required if c not in present]
    unexpected = [c for c in present if c not in required]

    print(f'{dataset_name} class folders: {present}')
    if missing:
        print(f'WARNING: {dataset_name} missing class folders: {missing}')
    if unexpected:
        print(f'WARNING: {dataset_name} has unexpected folders: {unexpected}')
    if not missing and not unexpected:
        print(f'OK: {dataset_name} class names match exactly.')
    return not missing and not unexpected

def print_counts(title: str, counts):
    print(title)
    total = 0
    for name, count in counts.items():
        total += count
        print(f'  {name}: {count}')
    print(f'  TOTAL: {total}')

def display_images(paths, title, max_images=4):
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        print(f'WARNING: No images available for {title}')
        return

    sample = paths[:max_images]
    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, path in zip(axes, sample):
        img = Image.open(path).convert('RGB')
        ax.imshow(img, cmap='gray')
        ax.set_title(path.name[:32])
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 6. Verify Required Real Datasets

The real labeled and real unlabeled datasets are required for the report-aligned experiments. Missing paths are reported clearly.

In [ ]:
critical_missing = []

for name, path in [
    ('real labeled dataset', REAL_LABELED_DIR),
    ('real unlabeled dataset', REAL_UNLABELED_DIR),
]:
    if path.exists():
        print(f'OK: Found {name}: {path}')
    else:
        print(f'WARNING: Missing {name}: {path}')
        critical_missing.append((name, path))

if critical_missing:
    print('\nCRITICAL: Real datasets are missing. Fix these paths before training notebooks.')
else:
    print('\nOK: Required real dataset paths exist.')

## 7. Verify Real Labeled Dataset

Check class names and count images per class for `data/processed/labelled_4232`.

In [ ]:
real_classes_ok = validate_class_names(REAL_LABELED_DIR, 'real labeled dataset')
real_labeled_counts = count_class_images(REAL_LABELED_DIR)
print_counts('Real labeled image counts:', real_labeled_counts)

## 8. Verify Real Unlabeled Dataset

Count all images under `data/processed/unlabelled_16934`. This dataset is used for COVID-QU contrastive pretraining.

In [ ]:
unlabeled_images = find_images(REAL_UNLABELED_DIR)
print(f'Real unlabeled image count: {len(unlabeled_images)}')
if len(unlabeled_images) == 0:
    print('WARNING: No unlabeled images found. Check REAL_UNLABELED_DIR.')

## 9. Verify Synthetic Datasets

Synthetic data is optional for this setup check because the Drive folders may not be configured yet. If present, class names and image counts are checked for both DCGAN and ACGAN outputs.

In [ ]:
def check_synthetic_dataset(dataset_root: Path, dataset_name: str):
    if dataset_root.exists():
        classes_ok = validate_class_names(dataset_root, dataset_name)
        counts = count_class_images(dataset_root)
        print_counts(f'{dataset_name} image counts:', counts)
    else:
        classes_ok = False
        counts = OrderedDict((name, 0) for name in CLASS_NAMES)
        print(f'WARNING: {dataset_name} path does not exist: {dataset_root}')
    return classes_ok, counts

dcgan_classes_ok, dcgan_counts = check_synthetic_dataset(DCGAN_SYNTHETIC_DIR, 'DCGAN synthetic dataset')
print()
acgan_classes_ok, acgan_counts = check_synthetic_dataset(ACGAN_SYNTHETIC_DIR, 'ACGAN synthetic dataset')

if not DCGAN_SYNTHETIC_DIR.exists():
    print('\nSet DCGAN_SYNTHETIC_DIR before running COVID-QU-Syn downstream experiments.')
if not ACGAN_SYNTHETIC_DIR.exists():
    print('Set ACGAN_SYNTHETIC_DIR if you want Stage 1 ACGAN-vs-DCGAN quality comparison.')

## 10. Display Real Labeled Samples

Show a few samples from each real labeled class.

In [ ]:
random.seed(42)
for class_name in CLASS_NAMES:
    images = find_images(class_image_dir(REAL_LABELED_DIR, class_name))
    random.shuffle(images)
    display_images(images, f'Real labeled: {class_name}', max_images=4)

## 11. Display Unlabeled Samples

Show several samples from the unlabeled COVID-QU pretraining dataset.

In [ ]:
random.seed(42)
unlabeled_preview = list(unlabeled_images)
random.shuffle(unlabeled_preview)
display_images(unlabeled_preview, 'Real unlabeled samples', max_images=6)

## 12. Display Synthetic Samples

If configured, show a few samples from each DCGAN and ACGAN synthetic class.

In [ ]:
def display_synthetic_samples(dataset_root: Path, dataset_name: str):
    if not dataset_root.exists():
        print(f'Skipping {dataset_name} previews because the path is missing: {dataset_root}')
        return
    random.seed(42)
    for class_name in CLASS_NAMES:
        images = find_images(class_image_dir(dataset_root, class_name))
        random.shuffle(images)
        display_images(images, f'{dataset_name}: {class_name}', max_images=4)

display_synthetic_samples(DCGAN_SYNTHETIC_DIR, 'DCGAN synthetic')
display_synthetic_samples(ACGAN_SYNTHETIC_DIR, 'ACGAN synthetic')

## 13. Summary

Review this summary before moving to training notebooks.

In [ ]:
print('Dataset check summary')
print('---------------------')
print(f'Real labeled path exists: {REAL_LABELED_DIR.exists()}')
print(f'Real labeled class names exact: {real_classes_ok}')
print(f'Real labeled total: {sum(real_labeled_counts.values())}')
print(f'Real unlabeled path exists: {REAL_UNLABELED_DIR.exists()}')
print(f'Real unlabeled total: {len(unlabeled_images)}')
print(f'DCGAN synthetic path exists: {DCGAN_SYNTHETIC_DIR.exists()}')
print(f'DCGAN synthetic class names exact: {dcgan_classes_ok}')
print(f'DCGAN synthetic total: {sum(dcgan_counts.values())}')
print(f'ACGAN synthetic path exists: {ACGAN_SYNTHETIC_DIR.exists()}')
print(f'ACGAN synthetic class names exact: {acgan_classes_ok}')
print(f'ACGAN synthetic total: {sum(acgan_counts.values())}')
print(f'Output root: {OUTPUT_ROOT}')

if not REAL_LABELED_DIR.exists() or not REAL_UNLABELED_DIR.exists():
    print('\nCRITICAL: Fix real dataset paths before training.')
elif not real_classes_ok:
    print('\nWARNING: Real labeled class folders do not match expected names exactly.')
elif not DCGAN_SYNTHETIC_DIR.exists():
    print('\nWARNING: DCGAN synthetic path is not configured yet. Real-data experiments can still proceed.')
elif not dcgan_classes_ok:
    print('\nWARNING: DCGAN synthetic class folders do not match expected names exactly.')
elif ACGAN_SYNTHETIC_DIR.exists() and not acgan_classes_ok:
    print('\nWARNING: ACGAN synthetic class folders do not match expected names exactly.')
else:
    print('\nOK: Required data paths and DCGAN class folders are ready for the 12-experiment pipeline.')